In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (
    SparkSession.builder
    .appName("dd")
    .master("local[*]") 
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

your 131072x1 screen size is bogus. expect trouble
25/11/28 00:43:12 WARN Utils: Your hostname, DESKTOP-OPG4HMT resolves to a loopback address: 127.0.1.1; using 172.22.209.89 instead (on interface eth0)
25/11/28 00:43:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/28 00:43:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df=spark.read.csv('/home/jiinss/data_project/data/2019-Oct.csv', header=True, inferSchema=True)
df.show()

+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 09:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 09:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 09:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 09:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 09:00:04|      view|   1004237|2053013555631882655|electr

In [ ]:
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



In [ ]:
#결측치 확인
df.select([
    F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|  brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+
|         0|         0|         0|          0|     13515609|6113008|    0|      0|           2|
+----------+----------+----------+-----------+-------------+-------+-----+-------+------------+



In [ ]:
len_df=df.count()
na_percent1,na_parcent2=100*13515609/len_df,100*6113008/len_df
print('category_code','brand')
print(na_percent1,na_parcent2)

category_code brand
31.839817526842477 14.400909293848933


In [5]:
#결측치처리 no_code, no_brand
df_f=df.fillna({'category_code':'no_code','brand':'no_brand','user_session':'no_session'})

In [6]:
#oct 데이터
#utc->kst->yymmdd
#day,week

df_t = df_f.withColumn("event_time_kst", F.from_utc_timestamp("event_time", "Asia/Seoul")) \
    .withColumn("date", F.to_date("event_time_kst")) \
    .withColumn("week", F.weekofyear("event_time_kst")) \
    .withColumn("day", F.dayofmonth("event_time_kst"))


In [ ]:
#dau
dau=df_t.groupBy("day").agg(F.countDistinct("user_id").alias("dau"))
dau.orderBy("day").show()
dau.orderBy(F.desc("dau")).limit(1).show()
dau.count()

+---+------+
|day|   dau|
+---+------+
|  1|201347|
|  2|196678|
|  3|181067|
|  4|175951|
|  5|209754|
|  6|193373|
|  7|197484|
|  8|189993|
|  9|223954|
| 10|214216|
| 11|212302|
| 12|226069|
| 13|205832|
| 14|228061|
| 15|211084|
| 16|231692|
| 17|228152|
| 18|209981|
| 19|233277|
| 20|226708|
+---+------+
only showing top 20 rows



+---+------+
|day|   dau|
+---+------+
| 19|233277|
+---+------+



31

In [ ]:
funnel_raw = df_t.groupBy("user_session", "event_type") \
               .agg(F.min("event_time_kst").alias("first_time"))

from pyspark.sql import Window
w = Window.partitionBy("user_session")

funnel_p = (
    df_t
        .withColumn("view",F.min(F.when(F.col("event_type") == "view",F.col("event_time_kst"))).over(w))
        .withColumn("cart",F.min(F.when(F.col("event_type") == "cart",F.col("event_time_kst"))).over(w))
        .withColumn("purchase",F.min(F.when(F.col("event_type") == "purchase", F.col("event_time_kst"))).over(w))
        .select("user_session", "view", "cart", "purchase")
        .distinct()
)


In [ ]:
funnel_p.show()

+--------------------+-------------------+-------------------+-------------------+
|        user_session|               view|               cart|           purchase|
+--------------------+-------------------+-------------------+-------------------+
|000081ea-9376-4eb...|2019-10-25 03:05:44|2019-10-25 03:06:14|2019-10-25 03:08:58|
|000174ac-0ea3-402...|2019-10-19 04:44:59|2019-10-19 04:45:11|2019-10-19 04:45:25|
|0002b07c-85cd-46e...|2019-10-15 03:14:45|2019-10-15 03:16:48|2019-10-15 03:20:16|
|00040234-87dc-459...|2019-10-11 11:17:59|2019-10-11 11:18:12|2019-10-11 11:19:32|
|0004400f-dc39-410...|2019-10-16 23:22:52|2019-10-16 23:24:18|2019-10-16 23:24:33|
|0004c309-ff34-44b...|2019-10-14 05:53:30|2019-10-14 05:55:25|2019-10-14 05:56:47|
|000941cc-a55d-4a5...|2019-10-25 14:08:55|2019-10-25 14:17:49|2019-10-25 14:20:26|
|000a2754-1167-47c...|2019-10-29 05:35:29|2019-10-29 05:36:20|2019-10-29 05:56:13|
|000e2bba-bbf2-432...|2019-10-11 02:51:25|2019-10-11 02:51:33|2019-10-11 02:51:47|
|001